# 07. 빈차 효율 시뮬레이션

택시의 하차 후 다음 승차까지의 빈차시간/빈차거리를 분석하고,
최적 재배치 시뮬레이션을 통해 빈차 효율 개선 가능성을 추정한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac: plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
df = pd.read_csv('./DC_TBYXD012.csv')
print(f'전체 건수: {len(df):,}')

# 시간 파싱
df['RIDE_DT'] = pd.to_datetime(df['RIDE_DTIME'], format='%Y%m%d%H%M%S')
df['ALIGHT_DT'] = pd.to_datetime(df['ALIGHT_DTIME'], format='%Y%m%d%H%M%S')
df['RIDE_HOUR'] = df['RIDE_DT'].dt.hour
df['ALIGHT_HOUR'] = df['ALIGHT_DT'].dt.hour

df.head()

## 1. 연속 운행 탐지: 빈차시간 계산

같은 차량(TAXI_VEHC_ID)의 운행을 하차시간 기준 정렬 후,
이전 운행의 하차시간과 다음 운행의 승차시간 차이를 빈차시간으로 정의한다.

In [ ]:
# 차량별 승차시간 정렬
df = df.sort_values(['TAXI_VEHC_ID', 'RIDE_DT']).reset_index(drop=True)

# 같은 차량의 이전 운행 하차시간
df['PREV_ALIGHT_DT'] = df.groupby('TAXI_VEHC_ID')['ALIGHT_DT'].shift(1)
df['PREV_ALIGHT_A_CD'] = df.groupby('TAXI_VEHC_ID')['ALIGHT_A_CD'].shift(1)
df['PREV_ALIGHT_POS_X'] = df.groupby('TAXI_VEHC_ID')['ALIGHT_POS_X'].shift(1)
df['PREV_ALIGHT_POS_Y'] = df.groupby('TAXI_VEHC_ID')['ALIGHT_POS_Y'].shift(1)

# 빈차시간 (분)
df['VACANCY_MIN'] = (df['RIDE_DT'] - df['PREV_ALIGHT_DT']).dt.total_seconds() / 60

# 비합리적 빈차시간 제외 (0분 미만 또는 480분(8시간) 초과 → 연속 운행이 아닌 것으로 판단)
vacancy = df[(df['VACANCY_MIN'] > 0) & (df['VACANCY_MIN'] <= 480)].copy()

print(f'유효 빈차 구간 수: {len(vacancy):,}')
print(f'평균 빈차시간: {vacancy["VACANCY_MIN"].mean():.1f}분')
print(f'중앙값 빈차시간: {vacancy["VACANCY_MIN"].median():.1f}분')

## 2. 빈차시간 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 히스토그램
axes[0].hist(vacancy['VACANCY_MIN'], bins=100, color='#5C6BC0', edgecolor='white', alpha=0.8)
axes[0].set_title('빈차시간 분포', fontsize=13)
axes[0].set_xlabel('빈차시간 (분)')
axes[0].set_ylabel('건수')
axes[0].axvline(vacancy['VACANCY_MIN'].median(), color='red', linestyle='--', label=f'중앙값 {vacancy["VACANCY_MIN"].median():.0f}분')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 시간대별 평균 빈차시간
hourly_vacancy = vacancy.groupby('ALIGHT_HOUR')['VACANCY_MIN'].mean()
axes[1].bar(hourly_vacancy.index, hourly_vacancy.values, color='#FF7043')
axes[1].set_title('시간대별 평균 빈차시간', fontsize=13)
axes[1].set_xlabel('하차 시간대')
axes[1].set_ylabel('평균 빈차시간 (분)')
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.show()

## 3. 빈차거리(VACNTV_DIST) 분포 및 시간대별 평균

In [ ]:
# 빈차거리가 있는 건만 사용
vac_dist = df[df['VACNTV_DIST'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 히스토그램
axes[0].hist(vac_dist['VACNTV_DIST'], bins=100, color='#26A69A', edgecolor='white', alpha=0.8)
axes[0].set_title('빈차거리(VACNTV_DIST) 분포', fontsize=13)
axes[0].set_xlabel('빈차거리 (m)')
axes[0].set_ylabel('건수')
axes[0].axvline(vac_dist['VACNTV_DIST'].median(), color='red', linestyle='--',
                label=f'중앙값 {vac_dist["VACNTV_DIST"].median():,.0f}m')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 시간대별 평균 빈차거리
hourly_dist = vac_dist.groupby('RIDE_HOUR')['VACNTV_DIST'].mean()
axes[1].bar(hourly_dist.index, hourly_dist.values, color='#AB47BC')
axes[1].set_title('시간대별 평균 빈차거리', fontsize=13)
axes[1].set_xlabel('승차 시간대')
axes[1].set_ylabel('평균 빈차거리 (m)')
axes[1].set_xticks(range(24))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print(f'전체 평균 빈차거리: {vac_dist["VACNTV_DIST"].mean():,.0f}m')
print(f'전체 중앙값 빈차거리: {vac_dist["VACNTV_DIST"].median():,.0f}m')

## 4. 비효율 구간 식별: 빈차시간 상위 10%의 하차 위치

In [ ]:
# 빈차시간 상위 10% 기준
threshold = vacancy['VACANCY_MIN'].quantile(0.9)
print(f'빈차시간 상위 10% 기준: {threshold:.1f}분')

top10pct = vacancy[vacancy['VACANCY_MIN'] >= threshold].copy()
print(f'상위 10% 건수: {len(top10pct):,}')

# 하차 행정동별 비효율 건수
inefficient_areas = top10pct['PREV_ALIGHT_A_CD'].value_counts().head(20).reset_index()
inefficient_areas.columns = ['행정동코드', '비효율건수']

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(inefficient_areas['행정동코드'].astype(str), inefficient_areas['비효율건수'], color='#EF5350')
ax.set_title(f'빈차시간 상위 10% (>= {threshold:.0f}분) 건의 하차 행정동 Top 20', fontsize=13)
ax.set_xlabel('비효율 건수')
ax.set_ylabel('하차 행정동 코드')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

# 비효율 상위 행정동의 평균 빈차시간
top_areas = top10pct.groupby('PREV_ALIGHT_A_CD').agg(
    건수=('VACANCY_MIN', 'size'),
    평균빈차시간=('VACANCY_MIN', 'mean')
).nlargest(10, '건수')
print('\n비효율 상위 10 행정동 상세:')
print(top_areas.round(1).to_string())

## 5. 행정동별 빈차 효율 지수

In [ ]:
# 하차 행정동 기준으로 평균 빈차시간, 평균 빈차거리 계산
# vacancy 데이터에 빈차거리 병합
vacancy_with_dist = vacancy[vacancy['VACNTV_DIST'] > 0].copy()

area_efficiency = vacancy_with_dist.groupby('PREV_ALIGHT_A_CD').agg(
    건수=('VACANCY_MIN', 'size'),
    평균빈차시간=('VACANCY_MIN', 'mean'),
    평균빈차거리=('VACNTV_DIST', 'mean')
).reset_index()

# 최소 건수 필터
area_efficiency = area_efficiency[area_efficiency['건수'] >= 30].copy()
area_efficiency.columns = ['행정동코드', '건수', '평균빈차시간(분)', '평균빈차거리(m)']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 빈차시간 기준 비효율 Top 15
worst_time = area_efficiency.nlargest(15, '평균빈차시간(분)')
axes[0].barh(worst_time['행정동코드'].astype(str), worst_time['평균빈차시간(분)'], color='#EF5350')
axes[0].set_title('평균 빈차시간 상위 15 행정동', fontsize=13)
axes[0].set_xlabel('평균 빈차시간 (분)')
axes[0].invert_yaxis()

# 빈차거리 기준 비효율 Top 15
worst_dist = area_efficiency.nlargest(15, '평균빈차거리(m)')
axes[1].barh(worst_dist['행정동코드'].astype(str), worst_dist['평균빈차거리(m)'], color='#FF7043')
axes[1].set_title('평균 빈차거리 상위 15 행정동', fontsize=13)
axes[1].set_xlabel('평균 빈차거리 (m)')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

# 산점도: 빈차시간 vs 빈차거리
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    area_efficiency['평균빈차시간(분)'],
    area_efficiency['평균빈차거리(m)'],
    s=area_efficiency['건수'] / area_efficiency['건수'].max() * 200,
    alpha=0.5, color='#5C6BC0', edgecolors='white'
)
ax.set_title('행정동별 빈차 효율 지수 (크기=건수)', fontsize=14)
ax.set_xlabel('평균 빈차시간 (분)')
ax.set_ylabel('평균 빈차거리 (m)')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 6. 최적 재배치 시뮬레이션

빈차시간 상위 건에 대해 "가장 가까운 수요 핫스팟으로 이동했으면?" 가정 아래
절감 가능한 빈차시간을 추정한다.

**로직:**
1. 행정동별 승차 건수로 수요 핫스팟 상위 20개 선정
2. 빈차시간 상위 10% 건의 하차 위치에서 가장 가까운 핫스팟까지 거리 계산
3. 평균 택시 속도(30km/h) 가정 시 이동시간 추정
4. 기존 빈차시간 vs 핫스팟 이동시간 비교

In [ ]:
from scipy.spatial.distance import cdist

# 수요 핫스팟: 승차 행정동 기준 상위 20개의 평균 좌표
ride_counts = df.groupby('RIDE_A_CD').size().reset_index(name='수요건수')
top_hotspots = ride_counts.nlargest(20, '수요건수')

# 핫스팟별 평균 좌표
hotspot_coords = df[df['RIDE_A_CD'].isin(top_hotspots['RIDE_A_CD'])].groupby('RIDE_A_CD').agg(
    LAT=('RIDE_POS_Y', 'mean'),
    LON=('RIDE_POS_X', 'mean')
).reset_index()

# 좌표 변환 (위도*10000000 → 도)
hotspot_coords['LAT_DEG'] = hotspot_coords['LAT'] / 10000000
hotspot_coords['LON_DEG'] = hotspot_coords['LON'] / 10000000

print(f'수요 핫스팟 {len(hotspot_coords)}개 선정')
print(hotspot_coords[['RIDE_A_CD', 'LAT_DEG', 'LON_DEG']].to_string(index=False))

In [ ]:
# 빈차시간 상위 10% 건의 하차 좌표
sim_data = top10pct[['PREV_ALIGHT_POS_X', 'PREV_ALIGHT_POS_Y', 'VACANCY_MIN']].dropna().copy()
sim_data['ALIGHT_LAT'] = sim_data['PREV_ALIGHT_POS_Y'] / 10000000
sim_data['ALIGHT_LON'] = sim_data['PREV_ALIGHT_POS_X'] / 10000000

# 유효 좌표만 (한국 범위 대략 필터)
sim_data = sim_data[
    (sim_data['ALIGHT_LAT'] > 33) & (sim_data['ALIGHT_LAT'] < 39) &
    (sim_data['ALIGHT_LON'] > 124) & (sim_data['ALIGHT_LON'] < 132)
].reset_index(drop=True)

print(f'시뮬레이션 대상 건수: {len(sim_data):,}')

# 각 하차 위치에서 가장 가까운 핫스팟까지 거리 (Haversine 근사: 위경도 차이 * 111km)
alight_coords = sim_data[['ALIGHT_LAT', 'ALIGHT_LON']].values
hot_coords = hotspot_coords[['LAT_DEG', 'LON_DEG']].values

# 도 단위 거리 → km 변환 (간단 근사)
def approx_dist_km(p1, p2):
    """위경도 두 점 간 근사 거리 (km)"""
    dlat = (p1[0] - p2[0]) * 111
    dlon = (p1[1] - p2[1]) * 111 * np.cos(np.radians((p1[0] + p2[0]) / 2))
    return np.sqrt(dlat**2 + dlon**2)

# 각 하차 위치에서 최근접 핫스팟까지 거리
min_dists = []
for i in range(len(alight_coords)):
    dists = [approx_dist_km(alight_coords[i], hot_coords[j]) for j in range(len(hot_coords))]
    min_dists.append(min(dists))

sim_data['NEAREST_HOTSPOT_KM'] = min_dists

# 30km/h 가정 시 이동시간 (분)
sim_data['MOVE_TO_HOTSPOT_MIN'] = sim_data['NEAREST_HOTSPOT_KM'] / 30 * 60

# 절감 가능 빈차시간 = 기존 빈차시간 - 핫스팟 이동시간 (음수면 0)
sim_data['SAVED_MIN'] = (sim_data['VACANCY_MIN'] - sim_data['MOVE_TO_HOTSPOT_MIN']).clip(lower=0)

print(f'\n--- 시뮬레이션 결과 ---')
print(f'평균 기존 빈차시간: {sim_data["VACANCY_MIN"].mean():.1f}분')
print(f'평균 핫스팟 이동시간: {sim_data["MOVE_TO_HOTSPOT_MIN"].mean():.1f}분')
print(f'평균 절감 가능 시간: {sim_data["SAVED_MIN"].mean():.1f}분')
print(f'총 절감 가능 시간: {sim_data["SAVED_MIN"].sum():,.0f}분 ({sim_data["SAVED_MIN"].sum()/60:,.0f}시간)')
print(f'절감율: {sim_data["SAVED_MIN"].sum() / sim_data["VACANCY_MIN"].sum() * 100:.1f}%')

In [ ]:
# 시뮬레이션 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 기존 빈차시간 vs 핫스팟 이동시간 분포
axes[0].hist(sim_data['VACANCY_MIN'], bins=50, alpha=0.6, label='기존 빈차시간', color='#EF5350')
axes[0].hist(sim_data['MOVE_TO_HOTSPOT_MIN'], bins=50, alpha=0.6, label='핫스팟 이동시간', color='#4CAF50')
axes[0].set_title('기존 빈차시간 vs 핫스팟 이동시간', fontsize=13)
axes[0].set_xlabel('시간 (분)')
axes[0].set_ylabel('건수')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 절감 가능 시간 분포
axes[1].hist(sim_data['SAVED_MIN'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[1].set_title('절감 가능 빈차시간 분포', fontsize=13)
axes[1].set_xlabel('절감 시간 (분)')
axes[1].set_ylabel('건수')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 7. 시간대별 빈차 효율 히트맵 (행정동 x 시간대)

In [ ]:
# 하차 행정동 x 하차 시간대별 평균 빈차시간
heatmap_data = vacancy.groupby(['PREV_ALIGHT_A_CD', 'ALIGHT_HOUR'])['VACANCY_MIN'].mean().reset_index()
heatmap_data.columns = ['행정동', '시간대', '평균빈차시간']

# 건수 기준 상위 30개 행정동만
top30_areas = vacancy['PREV_ALIGHT_A_CD'].value_counts().head(30).index
heatmap_filtered = heatmap_data[heatmap_data['행정동'].isin(top30_areas)]

pivot = heatmap_filtered.pivot_table(index='행정동', columns='시간대', values='평균빈차시간', fill_value=0)
pivot.index = pivot.index.astype(str)

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.5, fmt='.0f',
            cbar_kws={'label': '평균 빈차시간 (분)'})
ax.set_title('시간대별 빈차 효율 히트맵 (하차 행정동 상위 30 x 시간대)', fontsize=14)
ax.set_xlabel('하차 시간대')
ax.set_ylabel('하차 행정동 코드')
plt.tight_layout()
plt.show()

## 8. 요약

In [ ]:
print('=' * 60)
print('빈차 효율 시뮬레이션 요약')
print('=' * 60)
print(f'분석 대상 유효 빈차 구간: {len(vacancy):,}건')
print(f'전체 평균 빈차시간: {vacancy["VACANCY_MIN"].mean():.1f}분')
print(f'전체 중앙값 빈차시간: {vacancy["VACANCY_MIN"].median():.1f}분')
print(f'빈차시간 상위 10% 기준: {threshold:.1f}분')
print()
print(f'전체 평균 빈차거리: {vac_dist["VACNTV_DIST"].mean():,.0f}m')
print(f'전체 중앙값 빈차거리: {vac_dist["VACNTV_DIST"].median():,.0f}m')
print()
print('--- 재배치 시뮬레이션 (상위 10% 비효율 건) ---')
print(f'대상 건수: {len(sim_data):,}건')
print(f'평균 기존 빈차시간: {sim_data["VACANCY_MIN"].mean():.1f}분')
print(f'평균 핫스팟 이동시간: {sim_data["MOVE_TO_HOTSPOT_MIN"].mean():.1f}분')
print(f'평균 절감 가능: {sim_data["SAVED_MIN"].mean():.1f}분')
print(f'총 절감 가능: {sim_data["SAVED_MIN"].sum()/60:,.0f}시간')
print(f'절감율: {sim_data["SAVED_MIN"].sum() / sim_data["VACANCY_MIN"].sum() * 100:.1f}%')
print('=' * 60)

## 분석 결과 해석

- **빈차시간 분포**: 대부분의 빈차는 단시간이나, 상위 10%에서 장시간 빈차 발생
- **시간대별 패턴**: 새벽/심야 시간대 빈차시간이 길어지는 경향
- **비효율 지역**: 특정 행정동에서 빈차시간이 집중적으로 길게 나타남
- **재배치 시뮬레이션**: 수요 핫스팟으로의 사전 이동 시 상당한 빈차시간 절감 가능
- **정책 시사점**: 빈차 효율이 낮은 지역/시간대에 대한 택시 배차 최적화 필요